# Processing ISFS 5-Minute Averaged Data

This notebook processes M2HATS ISFS 5-minute averaged (geo-rotated, tilt-corrected) NetCDF files from the surface network into a consolidated, analysis-ready Zarr store. It works in tandem with [`code/m2hats_5min_to_zarr.py`](../../code/m2hats_5min_to_zarr.py), which defines `restructure_5min`, the preprocessing helpers (`fix_t1`, `fill_missing_vars`), and the variable metadata used in the final output.

[`code/m2hats_5min_to_zarr.py`](../../code/m2hats_5min_to_zarr.py) reuses the conventions of the high-rate workflow (`ISFS_highrate_geo.ipynb` / [`code/m2hats_to_zarr.py`](../../code/m2hats_to_zarr.py)) so both stores read identically:
- **Variable naming** — ISFS short names are mapped to CF-style names via `VAR_METADATA` (e.g. `u` → `wind_u`, `h2o` → `water_vapor_density`), with the original kept in `isfs_short_name` / `isfs_long_name`
- **Sites** — `t0` is the multi-level profile tower and appears only under `/profile_t0`; the horizontal array uses `t0p`, `t1`…`t49`, with the same `site_height` / `site_lon` / `site_lat` / `csat_model_appendix_a` coordinates
- **t1 sonic swap** — pre-swap (CSAT3) data is labelled `t1p` and post-swap (CSAT3A) data `t1`, each NaN-padded outside its period (swap at 2023-08-23T13:00Z)
- **Down times** — t0 data are *not* masked; every group carries a CF `valid` flag built from `T0_PROFILE_VALID_START` and `T0_LOWERING_WINDOWS` (0 = `ok`, 1 = `t0_pre_31july_relocation`, 2 = `t0_tower_lowered`, 3 = `t0_battery_swap`)

**Key differences from the high-rate workflow:**
- No `sample` dimension or `sample_offset` — each variable is one value per 5-minute bin (timestamps are bin centres), plus 2nd- and 3rd-order moments
- One file per day; timestamps are absolute, so no `base_time` reconstruction is needed
- Moments are named `cov_<a>_<b>` (2nd order) and `m3_<a>_<b>_<c>` (3rd order) from their ISFS component tokens, lower-cased (`u, v, w, tc, h2o, co2, p`; e.g. ISFS `w_P` → `cov_w_p`)
- `time` carries CF `standard_name`/`long_name`/`axis` attrs marking 5-minute bin centres; `tilt_corrected` and the store `title` are taken from the source `wind3d_tilt_correction` attr
- Additional `soils/` and `radiation/` groups for sensors that exist only in the 5-minute product

**Workflow:**
1. **Setup** — Python imports, path configuration, and Dask cluster on Casper
2. **Inspect one file** — Sanity-check variable count, naming patterns, and fill-value encoding before batch processing
3. **Build the file list** — Assemble the sorted source-file list and a campaign-wide variable inventory
4. **Write the combined Zarr store** — Concatenate all daily files into a single Zarr store, with the t1 swap split and missing-variable padding (resume-safe)
5. **Validate the combined store** — Spot-check structure and values before restructuring
6. **Restructure into a DataTree** — Reorganize from a flat variable-per-site layout into a hierarchical DataTree with instrument groups and site/height coordinates
7. **Zarr format evaluation** — Benchmark read performance and compare compressed size against the source NetCDF
8. **Verify the output** — Inspect the store and compare a time slice against the raw NetCDF source

**Input:** `surface_tiltcor_5min/*.nc`  
**Output:** `GDEX_datasets/isfs_m2hats_qc_geo_tiltcor_5min.zarr`

## Stage 1: Setup

### Imports

In [19]:
import os
os.chdir("/glade/u/home/myasears/m2hats-in-gdex")

In [39]:
import sys
sys.path.insert(0, "/glade/u/home/myasears/m2hats-in-gdex/code")  # m2hats_*_to_zarr.py live in code/

import re
import time
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import zarr as zarr_lib

import m2hats_5min_to_zarr as m
from m2hats_5min_to_zarr import (
    restructure_5min,
    VAR_METADATA,
    fix_t1,
    fill_missing_vars,
    parse_var,
    classify,
    T1_SWAP_UTC,
    dir_size_mb,
)

### Configure paths

All data paths are set here. Update these if source files or output locations change.

In [21]:
# Source 5-minute averaged NetCDF files
isfs_5min_path = "/lustre/desc1/scratch/myasears/M2HATS/FDA_datasets/surface_tiltcor_5min"

# Intermediate combined Zarr store (Stage 4 output / Stages 5-6 input)
working_zarr = "/lustre/desc1/scratch/myasears/M2HATS/WORKING_datasets/isfs_m2hats_qc_geo_tiltcor_5min_combined.zarr"

# Final restructured Zarr store (Stage 6 output)
gdex_zarr = "/lustre/desc1/scratch/myasears/M2HATS/GDEX_datasets/isfs_m2hats_qc_geo_tiltcor_5min.zarr"

### Spin up a Dask cluster on Casper

The 5-minute dataset is much smaller than high-rate (~188 MB total compressed), so fewer workers are needed. Scale to 4 workers before running any computation; adjust upward if restructuring is slow. All file reads and Zarr writes are lazy Dask graphs, so the cluster must be running before any `.compute()` or `.to_zarr()` calls below.

In [22]:
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client

lustre_scratch = "/lustre/desc1/scratch/myasears"

cluster = PBSCluster(
    job_name = 'dask-eol-26',
    cores = 1,
    memory = '20GiB',
    processes = 1,
    local_directory = f"{lustre_scratch}/dask/spill/",
    log_directory = f"{lustre_scratch}/dask/logs/",
    resource_spec = 'select=1:ncpus=1:mem=8GB',
    queue = 'casper',
    walltime = '2:00:00',
    interface = 'ext'
)

client = Client(cluster)

cluster.scale(jobs=4)
client.wait_for_workers(4)

/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46237 instead
  warnings.warn(


In [23]:
cluster

PBSCluster(a6370346, 'tcp://128.117.208.99:40915', workers=4, threads=4, memory=32.00 GiB)

## Stage 2: Inspect one file

Open a single daily file to confirm the naming conventions, dimension layout, and fill-value encoding before committing to a full batch run. Unlike the high-rate data there is no known structural bug to fix.

> **Note on time encoding:** Unlike the high-rate files, 5-minute files store `time` as absolute `datetime64` bin centres (e.g. `00:02:30` for the 00:00–00:05 bin), so no `base_time` reconstruction is needed.

In [24]:
sample_file = sorted(Path(isfs_5min_path).glob("*.nc"))[2]  # pick an early-campaign day
ds_sample = xr.open_dataset(sample_file)

print(f"File     : {sample_file.name}")
print(f"Dims     : {dict(ds_sample.sizes)}")
print(f"Variables: {len(ds_sample.data_vars)}")
print(f"Time range: {ds_sample.time.values[0]}  →  {ds_sample.time.values[-1]}")
print(f"Time step : {pd.Timedelta(ds_sample.time.diff('time').values[0])}")
print()

# Confirm fill-value encoding (NIDAS writes 1e37 not NaN; converted to NaN in Stage 6)
scalar_var = "u_4m_t2"
enc = ds_sample[scalar_var].encoding
print(f"Fill-value encoding for '{scalar_var}': {enc.get('_FillValue', 'not set')}")
print(f"dtype: {ds_sample[scalar_var].dtype}")
print()

# Show naming-pattern examples
for prefix in ["u", "w_w", "w_tc", "w_h2o", "T", "h2o", "Gsoil", "Rsw_in"]:
    matches = [v for v in ds_sample.data_vars if v.startswith(f"{prefix}_")][:3]
    if matches:
        print(f"{prefix:10s}: {matches}")

ds_sample.close()

File     : isfs_m2hats_qc_geo_tiltcor_5min_20230725.nc
Dims     : {'time': 288, 'sites': 17}
Variables: 2248
Time range: 2023-07-25T00:02:30.000000000  →  2023-07-25T23:57:30.000000000
Time step : 0 days 00:05:00

Fill-value encoding for 'u_4m_t2': 9.999999933815813e+36
dtype: float32

u         : ['u_4m_t29', 'u_u__4m_t29', 'u_v__4m_t29']
w_w       : ['w_w__4m_t29', 'w_w_w__4m_t29', 'w_w_tc__4m_t29']
w_tc      : ['w_tc__4m_t29', 'w_tc_tc__4m_t29', 'w_tc__15m_t0']
w_h2o     : ['w_h2o__4m_t29', 'w_h2o__15m_t0', 'w_h2o__4m_t8']
T         : ['T_4m_t35', 'T_7m_t0', 'T_4m_t2']
h2o       : ['h2o_4m_t29', 'h2o_h2o__4m_t29', 'h2o_15m_t0']
Gsoil     : ['Gsoil_t23', 'Gsoil_t2', 'Gsoil_t49']
Rsw_in    : ['Rsw_in_t2', 'Rsw_in_t49']


## Stage 3: Build the full file list

Glob all `.nc` files from the source directory in sorted (chronological) order. The final file (2023-09-25) is missing some dimensions and has been omitted.

In [25]:
isfs_5min_files = [
    f for f in sorted(Path(isfs_5min_path).glob("*.nc"))
    if f.name != "isfs_m2hats_qc_geo_tiltcor_5min_20230925.nc"
]

### Inventory variable shapes across the campaign

Build a lookup of every variable's non-time dims and dtype so gaps can be NaN-filled (via `fill_missing_vars`, Stage 4) when a day is missing a variable that appears elsewhere in the campaign.

In [26]:
# Not every variable appears in every daily file (e.g. a sensor was added,
# removed, or renamed partway through the campaign). If a variable is simply
# absent from a day's file, appending that file to the combined Zarr store
# leaves that variable's array shorter than the others -- which is what
# produces the "conflicting sizes for dimension 'time'" error on readback.
#
# Build one inventory of every variable's non-time dims/dtype across the
# *whole* campaign so we can NaN-fill any variable missing from a given day.
# Note: dtype is per-variable, not hardcoded -- some variables (e.g.
# counts_4m_t0p) are float64 while most are float32, and NaN-filling with
# the wrong dtype would cause a mismatch when the real data later appends.
var_templates = {}
for f in isfs_5min_files:
    with xr.open_dataset(f) as ds_f:
        for v, da in ds_f.data_vars.items():
            if v not in var_templates:
                var_templates[v] = {
                    "dims": da.dims,
                    "other_dims": {d: da.sizes[d] for d in da.dims if d != "time"},
                    "dtype": da.dtype,
                }

print(f"Found {len(var_templates)} unique variables across all {len(isfs_5min_files)} files")

Found 2279 unique variables across all 64 files


## Stage 4: Write the combined Zarr store

Process files one day at a time: open each daily file, apply preprocessing, and append the result to the Zarr store. The loop checks what has already been written on startup so it can be safely re-run to resume after an interruption.

Each file is preprocessed by (both from `m2hats_5min_to_zarr`):
1. **`fix_t1`** — splits site t1 at the sonic swap (CSAT3 → CSAT3A on 2023-08-23T13:00Z), exactly as `fix_time_and_t1` does for high-rate: pre-swap data is labelled `t1p` and post-swap data `t1`, each NaN-padded outside its valid period.
2. **`fill_missing_vars`** — pads any variable absent from a given day's file with correctly shaped, correctly typed NaNs, using the variable inventory built in Stage 3. Without this, appending a day that's missing a variable leaves that variable's array shorter than the others, which raises a "conflicting sizes for dimension 'time'" error on readback.

Fill values and t0 down times are handled in Stage 6, as in the high-rate workflow: sentinels become NaN, and down times are recorded in each group's `valid` flag rather than masked.

> **Rebuilding:** a working store written by an earlier version of this notebook (t0 NaN-masked, no `t1p`) is not compatible. Delete `working_zarr` before re-running this stage.

In [27]:
def date_key(p):
    m = re.search(r'_(\d{8})\.nc$', p.name)
    return m.group(1) if m else "unknown"

# Detect which dates are already written so we can resume
try:
    existing = xr.open_zarr(working_zarr)
    last_written = str(existing.time.values[-1])[:10].replace("-", "")
    print(f"Resuming after {last_written}")
    existing.close()
    first_run = False
except (FileNotFoundError, zarr_lib.errors.GroupNotFoundError):
    last_written = None
    first_run = True

for nc_file in isfs_5min_files:
    date = date_key(nc_file)
    if last_written and date <= last_written:
        print(f"Skipping {date} (already written)")
        continue

    print(f"Processing {date}…")
    ds_day = xr.open_dataset(nc_file, chunks={"time": 288})
    ds_day = fix_t1(ds_day)
    ds_day = fill_missing_vars(ds_day, var_templates)

    if first_run:
        ds_day.to_zarr(working_zarr, mode="w", consolidated=True)
        first_run = False
    else:
        ds_day.to_zarr(working_zarr, append_dim="time")

    ds_day.close()
    print(f"  Done {date}")

Resuming after 20230924
Skipping 20230723 (already written)
Skipping 20230724 (already written)
Skipping 20230725 (already written)
Skipping 20230726 (already written)
Skipping 20230727 (already written)
Skipping 20230728 (already written)
Skipping 20230729 (already written)
Skipping 20230730 (already written)
Skipping 20230731 (already written)
Skipping 20230801 (already written)
Skipping 20230802 (already written)
Skipping 20230803 (already written)
Skipping 20230804 (already written)
Skipping 20230805 (already written)
Skipping 20230806 (already written)
Skipping 20230807 (already written)
Skipping 20230808 (already written)
Skipping 20230809 (already written)
Skipping 20230810 (already written)
Skipping 20230811 (already written)
Skipping 20230812 (already written)
Skipping 20230813 (already written)
Skipping 20230814 (already written)
Skipping 20230815 (already written)
Skipping 20230816 (already written)
Skipping 20230817 (already written)
Skipping 20230818 (already written)
Skip

## Stage 5: Validate the combined store

Reload the combined Zarr store and run structural checks before restructuring: confirm the expected number of time steps, spot-check values against a raw NetCDF file, and verify the t1 / t1p split.

In [28]:
ds = xr.open_zarr(working_zarr)

print(f"Dims     : {dict(ds.sizes)}")
print(f"Variables: {len(ds.data_vars)}")
print(f"Time range: {ds.time.values[0]}  →  {ds.time.values[-1]}")
print(f"Total steps: {ds.sizes['time']} (expect {len(isfs_5min_files) * 288})")

Dims     : {'time': 18432, 'sites': 17}
Variables: 2297
Time range: 2023-07-23T00:02:30.000000000  →  2023-09-24T23:57:30.000000000
Total steps: 18432 (expect 18432)


In [29]:
# Compare a single day against the raw NetCDF source
CHECK_DATE = "20230725"
CHECK_VARS = ["u_4m_t2", "w_w__4m_t2", "T_4m_t2", "h2o_4m_t2"]

# Raw
raw_file = Path(isfs_5min_path) / f"isfs_m2hats_qc_geo_tiltcor_5min_{CHECK_DATE}.nc"
ds_raw = xr.open_dataset(raw_file)

# Zarr
zarr_day = ds.sel(time=ds.time.dt.strftime("%Y%m%d") == CHECK_DATE)

print(f"Comparing {CHECK_DATE} against raw NetCDF ({raw_file.name})")
print(f"{'variable':25s}  {'max_diff':>12s}  {'match (<1e-5)':>14s}")
print("─" * 58)

for v in CHECK_VARS:
    if v not in ds_raw or v not in zarr_day:
        print(f"{v:25s}  (not in both datasets)")
        continue
    raw_vals  = ds_raw[v].values.astype("float64")
    zarr_vals = zarr_day[v].values.astype("float64")
    both_nan  = np.isnan(raw_vals) & np.isnan(zarr_vals)
    max_diff  = float(np.nanmax(np.abs(raw_vals - zarr_vals))) if not np.all(both_nan) else float("nan")
    match     = bool(np.all(both_nan | (np.abs(raw_vals - zarr_vals) < 1e-5)))
    print(f"{v:25s}  {max_diff:>12.2e}  {'✓' if match else '✗':>14s}")

ds_raw.close()

Comparing 20230725 against raw NetCDF (isfs_m2hats_qc_geo_tiltcor_5min_20230725.nc)
variable                       max_diff   match (<1e-5)
──────────────────────────────────────────────────────────
u_4m_t2                        0.00e+00               ✓
w_w__4m_t2                     0.00e+00               ✓
T_4m_t2                        0.00e+00               ✓
h2o_4m_t2                      0.00e+00               ✓


In [30]:
# ── Verify the t1 / t1p split ────────────────────────────────────────────────
# t1p (CSAT3) must be all-NaN after the swap and t1 (CSAT3A) all-NaN before.
pre  = ds.time < T1_SWAP_UTC
post = ~pre

print(f"t1 sonic swap at {T1_SWAP_UTC}")
print(f"{'variable':12s}  {'valid pre':>10s}  {'valid post':>11s}")
print("-" * 38)
for v in ["u_4m_t1p", "u_4m_t1"]:
    ok = ds[v].notnull()
    print(f"{v:12s}  {int(ok.where(pre, False).sum()):>10d}  {int(ok.where(post, False).sum()):>11d}")

t1 sonic swap at 2023-08-23T13:00:00
variable       valid pre   valid post
--------------------------------------
u_4m_t1p            8173            0
u_4m_t1                0         9233


## Stage 6: Restructure into an analysis-ready DataTree

The combined store inherits the flat ISFS variable-per-site naming (`u_4m_t2`, `u_4m_t5`, …). This stage reshapes that layout into a hierarchical DataTree where each instrument group is a node with `site` or `height` as an explicit coordinate dimension, using the same variable names, coordinates, and `valid` flag as the high-rate store.

### Group hierarchy

```
/
├── array/
│   ├── sonic           (site: 51, time: N)   — wind_u/v/w, sonic_temperature, wind_speed, wind_from_direction, sonic_qc_flag + cov_*/m3_* moments
│   ├── irga            (site: 16, time: N)   — water_vapor_density, co2_density, irga_* + cov_* moments
│   ├── trh             (site: 16, time: N)   — air_temperature, relative_humidity, trh_fan_speed
│   └── barometer       (site: 16, time: N)   — air_pressure
├── profile_t0/
│   ├── sonic           (height: 8, time: N)  — as array/sonic
│   ├── irga            (height: 8, time: N)  — as array/irga
│   ├── trh             (height: 8, time: N)  — as array/trh
│   └── barometer       (height: 8, time: N)  — air_pressure
├── soils/
│   ├── heat_flux       (site: 3, time: N)    — soil_heat_flux (t2, t23, t49)
│   ├── temperature     (site: 3, depth: 4, time: N) — soil_temperature
│   ├── moisture        (site: 3, time: N)    — soil_moisture
│   └── thermal_props   (site: 3, time: N)    — soil_thermal_conductivity, soil_probe_*
└── radiation/
    └── nr01            (site: 2, time: N)    — shortwave_in/out, thermopile_in/out, radiometer_case_temperature, leaf_wetness_voltage (t2, t49)
```

Every group also carries a `valid` flag on `time`. Array sites are `t0p`, `t1p`, `t1`…`t49` (sorted as in the high-rate store); `t0` appears only under `/profile_t0`.

> **2nd and 3rd moments** are included in the `sonic` and `irga` groups alongside their parent scalars. This keeps flux users from having to join across groups for the covariances they need most.

### Sanity-check the name parser

Confirm `classify` (built on the high-rate `parse_var`) routes flat variable names to the expected group, variable, and site/height before trusting it inside `restructure_5min`.

In [31]:
for test in ["u_4m_t2", "w_tc__4m_t29", "u_u_u__0_5m_t0", "u_4m_t1p", "Tsoil_3_1cm_t23", "Gsoil_t23", "Rsw_in_t49", "counts_4m_t2"]:
    print(f"{test:18s} →", classify(test))

u_4m_t2            → ('array/sonic', 'u', 't2', False)
w_tc__4m_t29       → ('array/sonic', 'w_tc', 't29', True)
u_u_u__0_5m_t0     → ('profile_t0/sonic', 'u_u_u', 0.5, True)
u_4m_t1p           → ('array/sonic', 'u', 't1p', False)
Tsoil_3_1cm_t23    → ('soils/temperature', 'Tsoil', (0.031, 't23'), False)
Gsoil_t23          → ('soils/heat_flux', 'Gsoil', 't23', False)
Rsw_in_t49         → ('radiation/nr01', 'Rsw_in', 't49', False)
counts_4m_t2       → None


### Preview the DataTree structure

Run `restructure_5min` in dry-run mode (`write=False`) on a small one-day subset to confirm the expected group hierarchy and dimension sizes without triggering a full computation.

In [32]:
subset = ds.isel(time=slice(0, 288)).persist()    # one day, in cluster memory

tree_test = restructure_5min(
    subset,
    output_path="./m2hats_5min_test.zarr",
    tilt_corrected=True,
    time_chunk=288,
    write=False,
)

for path in tree_test.groups:
    node = tree_test[path].ds
    if not node.data_vars:
        continue
    print(f"{path:25s} dims={dict(node.sizes)}")

/array/sonic              dims={'site': 51, 'time': 288}
/array/irga               dims={'site': 16, 'time': 288}
/array/trh                dims={'site': 16, 'time': 288}
/array/barometer          dims={'site': 16, 'time': 288}
/profile_t0/sonic         dims={'height': 8, 'time': 288}
/profile_t0/irga          dims={'height': 8, 'time': 288}
/profile_t0/trh           dims={'height': 8, 'time': 288}
/profile_t0/barometer     dims={'height': 8, 'time': 288}
/soils/heat_flux          dims={'site': 3, 'time': 288}
/soils/temperature        dims={'site': 3, 'depth': 4, 'time': 288}
/soils/moisture           dims={'site': 3, 'time': 288}
/soils/thermal_props      dims={'site': 3, 'time': 288}
/radiation/nr01           dims={'site': 2, 'time': 288}


Expected output:

```
/array/sonic              dims={'site': 51, 'time': 288}
/array/irga               dims={'site': 16, 'time': 288}
/array/trh                dims={'site': 16, 'time': 288}
/array/barometer          dims={'site': 16, 'time': 288}
/profile_t0/sonic         dims={'height': 8, 'time': 288}
/profile_t0/irga          dims={'height': 8, 'time': 288}
/profile_t0/trh           dims={'height': 8, 'time': 288}
/profile_t0/barometer     dims={'height': 8, 'time': 288}
/soils/heat_flux          dims={'site': 3, 'time': 288}
/soils/temperature        dims={'site': 3, 'depth': 4, 'time': 288}
/soils/moisture           dims={'site': 3, 'time': 288}
/soils/thermal_props      dims={'site': 3, 'time': 288}
/radiation/nr01           dims={'site': 2, 'time': 288}
```

Spot-check a few real values to be sure the data isn't all NaN:

In [33]:
leaf = tree_test["/array/sonic"].ds
print(leaf.wind_u.sel(site="t2").isel(time=slice(0, 5)).compute())

<xarray.DataArray 'wind_u' (time: 5)> Size: 20B
array([0.88300586, 0.06675231, 2.2745543 , 2.320093  , 1.478019  ],
      dtype=float32)
Coordinates:
  * time                   (time) datetime64[ns] 40B 2023-07-23T00:02:30 ... ...
    site                   <U2 8B 't2'
    site_height            float64 8B 4.539
    site_lon               float64 8B -117.1
    site_lat               float64 8B 38.04
    csat_model_appendix_a  <U12 48B 'CSAT3A+EC150'
Attributes:
    long_name:        Sonic wind U component
    units:            m s-1
    isfs_short_name:  u
    standard_name:    eastward_wind


## Stage 6 (cont.): Build the full DataTree

Call `restructure_5min` on the full combined store in dry-run mode (`write=False`) to build the DataTree in memory. The write step follows.

In [34]:
tree = restructure_5min(ds, gdex_zarr, write=False, tilt_corrected=True)

### Write the restructured Zarr store

Reload `m2hats_5min_to_zarr` to pick up any last-minute edits, stamp t1-swap metadata into the root group attributes, then write each leaf group to Zarr one at a time. Writing groups individually keeps each Dask graph small. The daily source files give 288-step Dask chunks, so each group is rechunked to `TIME_CHUNK` along `time` to match the Zarr encoding. After all groups are written, metadata is consolidated so the store can be opened efficiently without scanning every group.

In [35]:
importlib.reload(m)
from m2hats_5min_to_zarr import restructure_5min  # refresh the direct import too

TIME_CHUNK = 2016  # 5-minute steps (~1 week)

# Stamp the swap into root attrs so the store is self-documenting.
tree.attrs["t1_csat_swap_utc"] = "2023-08-23T13:00:00"
tree.attrs["t1_csat_swap_note"] = (
    "Site t1 sonic was a CSAT3 (30 Hz, site t1p) before this timestamp "
    "and a CSAT3A (60 Hz, site t1) after. Pre-swap data lives in "
    "/array/sonic under site t1p; post-swap data lives in "
    "/array/sonic under site t1. Each column is NaN-padded "
    "outside its valid period."
)

# Write root group (attrs only) first to initialise the store.
tree.ds.to_zarr(gdex_zarr, mode="w", consolidated=False)

# Write each leaf group separately — one Dask graph at a time.
for path in tree.groups:
    node = tree[path]
    if not node.ds.data_vars:
        continue
    enc = m._encoding_for(node.ds, TIME_CHUNK)
    # Align Dask chunks with Zarr encoding: non-time dims are written whole
    ds_w = node.ds.chunk({**{d: -1 for d in node.ds.dims if d != "time"}, "time": TIME_CHUNK})
    print(f"writing  {path:25s} ({ds_w.sizes.get('time', 0):>9,} steps)", flush=True)
    ds_w.to_zarr(gdex_zarr, group=path, mode="a", consolidated=False, encoding=enc)

zarr_lib.consolidate_metadata(gdex_zarr)
print("done.")

writing  /array/sonic              (   18,432 steps)


/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/group.py:2711: ZarrUserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=12, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/distributed/client.py:3374: UserWarning: Se

writing  /array/irga               (   18,432 steps)


/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/group.py:2711: ZarrUserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=12, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


writing  /array/trh                (   18,432 steps)


...

/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/group.py:2711: ZarrUserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(


writing  /radiation/nr01           (   18,432 steps)


/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/group.py:2711: ZarrUserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(


done.


/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=12, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


## Stage 7: Zarr format evaluation

Assess whether the Zarr store provides meaningful advantages over the source NetCDF files for the 5-minute dataset. Three metrics are measured:

1. **Compressed size** — compare the source NetCDF total against the Zarr store on disk
2. **Random-access read latency** — time a single-site, full-campaign extraction from both formats
3. **Columnar scan latency** — time reading one variable across all sites for the full campaign

> **Context for interpreting results:** The full 5-minute dataset is ~188 MB uncompressed NetCDF. For a dataset this small, Zarr's primary advantage is not raw compression ratio (both formats use efficient encodings) but rather **chunk-aligned random access** — pulling a single site or variable slice without reading unrelated data. The evaluation below quantifies this concretely.

In [36]:
# ── Disk size comparison ─────────────────────────────────────────────────────

nc_total_mb   = sum(os.path.getsize(f) for f in isfs_5min_files) / 1e6
zarr_total_mb = dir_size_mb(gdex_zarr)

print(f"Source NetCDF total  : {nc_total_mb:7.1f} MB")
print(f"Restructured Zarr    : {zarr_total_mb:7.1f} MB")
print(f"Compression ratio    : {nc_total_mb / zarr_total_mb:.2f}x")

Source NetCDF total  :   193.3 MB
Restructured Zarr    :   108.4 MB
Compression ratio    : 1.78x


In [37]:
# ── Random-access latency: single site, full campaign ────────────────────────

BENCH_SITE = "t23"
BENCH_VAR  = "wind_u"                              # Zarr name
BENCH_ISFS = "u"                                  # ISFS short name in the NetCDF

# Zarr
tree_bench = xr.open_datatree(gdex_zarr, engine="zarr", consolidated=True, chunks={})
t0 = time.perf_counter()
u_zarr = tree_bench["/array/sonic"].ds[BENCH_VAR].sel(site=BENCH_SITE).compute()
zarr_latency = time.perf_counter() - t0

# NetCDF (open all files and extract same variable)
t0 = time.perf_counter()
u_nc_parts = []
for f in isfs_5min_files:
    ds_nc = xr.open_dataset(f)
    vname = f"{BENCH_ISFS}_4m_{BENCH_SITE}"
    if vname in ds_nc:
        u_nc_parts.append(ds_nc[vname].values)
    ds_nc.close()
u_nc = np.concatenate(u_nc_parts) if u_nc_parts else np.array([])
nc_latency = time.perf_counter() - t0

print(f"Single-site full-campaign read of '{BENCH_VAR}' at {BENCH_SITE}:")
print(f"  Zarr   : {zarr_latency:.3f} s  ({len(u_zarr)} steps)")
print(f"  NetCDF : {nc_latency:.3f} s  ({len(u_nc)} steps)")
print(f"  Speedup: {nc_latency / zarr_latency:.1f}x" if zarr_latency > 0 else "")

Single-site full-campaign read of 'wind_u' at t23:
  Zarr   : 0.144 s  (18432 steps)
  NetCDF : 21.241 s  (18432 steps)
  Speedup: 147.7x


In [46]:
# ── Columnar scan: one variable, all sites, full campaign ────────────────────

# Zarr
t0 = time.perf_counter()
u_all_zarr = tree_bench["/array/sonic"].ds[BENCH_VAR].compute()
zarr_col_latency = time.perf_counter() - t0

# NetCDF
array_sites = [str(s) for s in u_all_zarr.site.values]
with xr.open_dataset(isfs_5min_files[0]) as ds_first:
    nc_vars = [f"{BENCH_ISFS}_4m_{s}" for s in array_sites if f"{BENCH_ISFS}_4m_{s}" in ds_first.data_vars]

t0 = time.perf_counter()
nc_col = xr.open_mfdataset(
    isfs_5min_files,
    combine="nested",
    concat_dim="time",
    coords="minimal",
    compat="override",
    data_vars=nc_vars,
    parallel=True,
).compute()
nc_col_latency = time.perf_counter() - t0

print(f"All-site full-campaign read of '{BENCH_VAR}':")
print(f"  Zarr   : {zarr_col_latency:.3f} s  shape={u_all_zarr.shape}")
print(f"  NetCDF : {nc_col_latency:.3f} s")
print(f"  Speedup: {nc_col_latency / zarr_col_latency:.1f}x" if zarr_col_latency > 0 else "")

/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 135.81 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


All-site full-campaign read of 'wind_u':
  Zarr   : 0.109 s  shape=(51, 18432)
  NetCDF : 652.240 s
  Speedup: 5973.2x


In [47]:
# ── Evaluation summary ───────────────────────────────────────────────────────

print("=" * 60)
print("Zarr format evaluation summary")
print("=" * 60)
print(f"  Dataset             : M2HATS ISFS 5-minute, 63 days")
print(f"  Time steps          : {ds.sizes['time']}")
print(f"  Variables           : {len(ds.data_vars)}")
print()
print(f"  Source NetCDF size  : {nc_total_mb:.1f} MB")
print(f"  Zarr store size     : {zarr_total_mb:.1f} MB")
print(f"  Compression ratio   : {nc_total_mb / zarr_total_mb:.2f}x")
print()
print(f"  Single-site read    : {nc_latency:.2f}s (NC) vs {zarr_latency:.2f}s (Zarr)")
print(f"  All-site read       : {nc_col_latency:.2f}s (NC) vs {zarr_col_latency:.2f}s (Zarr)")
print()
print("Assessment:")
print("  The 5-minute dataset is small enough that the raw NC files can be")
print("  concatenated in memory without Dask. Zarr provides the clearest")
print("  benefit for workflows that need (a) columnar access across all sites,")
print("  (b) the hierarchical DataTree structure for instrument-group selection,")
print("  or (c) lazy access that chains into larger multi-dataset analyses")
print("  (e.g. combining with the high-rate store for multi-timescale work).")
print("  If the primary use-case is simple per-day loading, the source NetCDF")
print("  files remain a perfectly efficient option.")

Zarr format evaluation summary
  Dataset             : M2HATS ISFS 5-minute, 63 days
  Time steps          : 18432
  Variables           : 2297

  Source NetCDF size  : 193.3 MB
  Zarr store size     : 108.4 MB
  Compression ratio   : 1.78x

  Single-site read    : 21.24s (NC) vs 0.14s (Zarr)
  All-site read       : 652.24s (NC) vs 0.11s (Zarr)

Assessment:
  The 5-minute dataset is small enough that the raw NC files can be
  concatenated in memory without Dask. Zarr provides the clearest
  benefit for workflows that need (a) columnar access across all sites,
  (b) the hierarchical DataTree structure for instrument-group selection,
  or (c) lazy access that chains into larger multi-dataset analyses
  (e.g. combining with the high-rate store for multi-timescale work).
  If the primary use-case is simple per-day loading, the source NetCDF
  files remain a perfectly efficient option.


## Stage 8: Verify the final output

Open the finished Zarr store as a DataTree and inspect the group structure as a sanity check on dimensions, coordinates, and data variables.

In [48]:
tree = xr.open_datatree(
    gdex_zarr,
    engine="zarr",
    consolidated=True,
    chunks={},
)

In [49]:
sonic = tree["/array/sonic"].ds

### Check the `valid` flag

Count time steps per flag value in `/profile_t0/sonic`. Expect flag 1 up to 2023-07-31, flag 2/3 during the `T0_LOWERING_WINDOWS`, and 0 everywhere in the `/array` groups.

In [50]:
valid = tree["/profile_t0/sonic"].ds["valid"].compute()
meanings = valid.attrs["flag_meanings"].split()
for value, meaning in zip(valid.attrs["flag_values"], meanings):
    print(f"{value}  {meaning:28s} {int((valid == value).sum()):>6d} steps")

0  ok                            15901 steps
1  t0_pre_31july_relocation       2220 steps
2  t0_tower_lowered                178 steps
3  t0_battery_swap                 133 steps


### Quantitative QA: compare Zarr output against raw NetCDF

For a single day and site, pull the same data from both the finished Zarr store and the original raw NetCDF file, then check that values agree to within floating-point tolerance (`< 1e-5`). Mismatches are flagged with `✗`. Each Zarr variable's `isfs_short_name` attribute gives the source name, so moments are checked too. Adjust `CHECK_DATE`, `CHECK_SITE`, and `CHECK_GROUP` to probe different instruments or periods (use site `t1p` before and `t1` after 2023-08-23T13:00Z).

In [51]:
# ── Parameters ────────────────────────────────────────────────────────────────
CHECK_DATE  = "20230725"
CHECK_SITE  = "t23"
CHECK_GROUP = "/array/sonic"
ISFS_PATH = Path(isfs_5min_path)

# ── 1. Zarr ───────────────────────────────────────────────────────────────────
group_ds = tree[CHECK_GROUP].ds
zarr_slice = (
    group_ds
    .sel(time=group_ds.time.dt.strftime("%Y%m%d") == CHECK_DATE, site=CHECK_SITE)
    .compute()
)

# ── 2. Raw NetCDF ─────────────────────────────────────────────────────────────
raw_file = ISFS_PATH / f"isfs_m2hats_qc_geo_tiltcor_5min_{CHECK_DATE}.nc"
raw_ds   = fix_t1(xr.open_dataset(raw_file))

# ── 3. Compare ────────────────────────────────────────────────────────────────
print(f"site={CHECK_SITE}  date={CHECK_DATE}")
print(f"zarr group : {CHECK_GROUP}")
print(f"raw file   : {raw_file.name}\n")
print(f"{'zarr var':30s}  {'raw var':25s}  {'match':>6s}  {'max_diff':>12s}")
print("─" * 80)

for zarr_var in sorted(zarr_slice.data_vars):
    isfs_short = zarr_slice[zarr_var].attrs.get("isfs_short_name")
    if isfs_short is None:
        continue
    candidates = [c for c in (f"{isfs_short}_4m_{CHECK_SITE}", f"{isfs_short}__4m_{CHECK_SITE}") if c in raw_ds]
    if not candidates:
        print(f"{zarr_var:30s}  {'(no match in raw)':25s}")
        continue
    raw_var = candidates[0]

    z = zarr_slice[zarr_var].values.astype("float64")
    r = raw_ds[raw_var].values.astype("float64")

    both_nan = np.isnan(z) & np.isnan(r)
    match    = bool(np.all(both_nan | (np.abs(z - r) < 1e-5)))
    max_diff = float(np.nanmax(np.abs(z - r))) if not np.all(both_nan) else float("nan")

    print(f"{zarr_var:30s}  {raw_var:25s}  {'✓' if match else '✗':>6s}  {max_diff:>12.2e}")

raw_ds.close()

site=t23  date=20230725
zarr group : /array/sonic
raw file   : isfs_m2hats_qc_geo_tiltcor_5min_20230725.nc

zarr var                        raw var                     match      max_diff
────────────────────────────────────────────────────────────────────────────────
cov_tc_tc                       tc_tc__4m_t23                   ✓      0.00e+00
cov_u_tc                        u_tc__4m_t23                    ✓      0.00e+00
cov_u_u                         u_u__4m_t23                     ✓      0.00e+00
cov_u_v                         u_v__4m_t23                     ✓      0.00e+00
cov_u_w                         u_w__4m_t23                     ✓      0.00e+00
cov_v_tc                        v_tc__4m_t23                    ✓      0.00e+00
cov_v_v                         v_v__4m_t23                     ✓      0.00e+00
cov_v_w                         v_w__4m_t23                     ✓      0.00e+00
cov_w_tc                        w_tc__4m_t23                    ✓      0.00e+00
cov_w_w    

## Cleanup

Shut down the Dask cluster and client to release PBS resources.

In [52]:
cluster.close()
client.close()